In [25]:
import os
import sys

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from xgboost import XGBRegressor


import matplotlib.pyplot as plt
from sklearn.linear_model import RANSACRegressor, LinearRegression

current_dir = os.getcwd()
project_root = current_dir[:current_dir.find("src") - 1]
sys.path.insert(0, project_root)
from src.models.filter_data.filter_data import *
from src.models.filter_data.feature_adder import *


In [26]:
temp_feature = "temperature"
csv_read_path = os.path.join(project_root, "data", "processed", "semi_processed.csv")

df = pd.read_csv(csv_read_path, encoding='utf-8')

In [27]:
csv_read_path = os.path.join(project_root, "data", "interim", "factors.csv")
df_factors = pd.read_csv(csv_read_path)

coefs = {}
grouped = df_factors.groupby(['PowerPlantCode', 'PowerPlantName', "UnitCode"])

for (pp_code, pp_name, unit_code), g in grouped:
    uniques = g[["a1IndexGas", "b1IndexGas"]].drop_duplicates()
    coefs[(pp_name, unit_code)] = []
    for row in uniques.itertuples(index=False):
        coefs[(pp_name, unit_code)].append((row.a1IndexGas, row.b1IndexGas))

In [61]:
def plot_with_lines(df, x_col, y_col, coefs, n, c, X, y_pred, title, save_path=None):
    """رسم نمودار پراکندگی با خطوط رگرسیون و نقاط پیش‌بینی"""
    
    # نمودار اصلی scatter
    fig = px.scatter(
        df,
        x=x_col,
        y=y_col,
        color="is_good_peak",
        title=title,
        labels={y_col: y_col.capitalize(), x_col: x_col.capitalize()},
        hover_data=['datetime', 'generation', x_col],
        size_max=1
    )

    # ظاهر نقاط
    fig.update_traces(
        marker=dict(size=4, sizemode='diameter', sizeref=1, opacity=0.7)
    )
    
    # خطوط رگرسیون
    for a, b in coefs.get((n, c), []):
        x_line = np.linspace(df[x_col].min(), df[x_col].max(), 100)
        y_line = a * x_line + b

        fig.add_trace(go.Scatter(
            x=x_line,
            y=y_line,
            mode="lines",
            name=f"y = {a:.3f}x + {b:.3f}",
            line=dict(dash="dash", width=2)
        ))

    # نقاط پیش‌بینی XGBoost (قابل خاموش/روشن شدن)
    fig.add_trace(go.Scatter(
        x=X.flatten(),
        y=y_pred,
        mode="markers",
        name="prediction",
        marker=dict(color="red", size=6, symbol="diamond", opacity=0.8),
        visible=True
    ))

    # ذخیره یا نمایش
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.write_html(save_path)
    else:
        fig.show()


def show(df_m1, save=False, param=None, ass=""):
    features = ["name", "code", "generation", f"{temp_feature}", "is_good_peak"]
    df_modified = df_m1[features].copy(deep=True)
    df_modified = df_modified[df_modified["is_good_peak"] >= 6]

    ds = Data_selector(df_modified)
    name, code = param['name'], param['code']
    one_unit_df = ds.filter_name_code(name, code)

    sens_temps = one_unit_df[temp_feature].values
    gens = one_unit_df['generation'].values
    X = sens_temps.reshape(-1, 1)
    y = gens

    # مدل XGBoost
    model = XGBRegressor(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )
    model.fit(X, y)
    y_pred = model.predict(X)

    # نمودار ۱: generation در زمان
    fig = px.scatter(
        df_m1,
        x="datetime",
        y='generation',
        color='is_good_peak',
        title='Generation over Time by Batch Interval',
        labels={'generation': 'Generation', 'datetime': 'Time'},
        hover_data=['datetime', 'generation', temp_feature]
    )
    
    fig.add_trace(go.Scatter(
        x=df_m1["datetime"],
        y=model.predict(df_m1["temperature"]),
        mode="lines",
        name="prediction",
        line=dict(color="red", width=2)
    ))    
    
    fig.add_trace(go.Scatter(
        x=df_m1["datetime"],
        y=df_m1["generation"],
        mode="lines",
        name="generation",
        line=dict(color="blue", width=2)
    ))    

    if save:
        path = f"{project_root}/src/visualization/unit_figs/filter6{ass}/{name}-{code}_l.html"
        os.makedirs(os.path.dirname(path), exist_ok=True)
        fig.write_html(path)
    else:
        fig.show()

    # نمودار ۲: generation vs temperature
    save_path = (f"{project_root}/src/visualization/unit_figs/filter6{ass}/{name}-{code}_s.html"
                 if save else None)
    plot_with_lines(
        df_m1, temp_feature, 'generation', coefs, name, code, X, y_pred,
        title='Generation vs Temperature', save_path=save_path
    )

    # نمودار ۳: declared vs temperature
    save_path = (f"{project_root}/src/visualization/unit_figs/filter6{ass}/{name}-{code}_d.html"
                 if save else None)
    plot_with_lines(
        df_m1, temp_feature, 'declared', coefs, name, code, X, y_pred,
        title='Declared vs Temperature', save_path=save_path
    )
    
    # نمودار ۴: declared vs generation
    fig = px.scatter(
        df_m1,
        x="declared",
        y='generation',
        color='is_good_peak',
        title='Declared over Generation by Batch Interval',
        labels={'generation': 'Generation', 'declared': 'Declared'},
        hover_data=['datetime', 'generation', temp_feature, "declared"]
    )

    x_line = np.linspace(0, 200, 100)
    y_line = x_line

    fig.add_trace(go.Scatter(
        x=x_line,
        y=y_line,
        mode="lines",
        name=f"y = x",
        line=dict(dash="dash", width=2)
    ))

    if save:
        path = f"{project_root}/src/visualization/unit_figs/filter6{ass}/{name}-{code}_dg.html"
        os.makedirs(os.path.dirname(path), exist_ok=True)
        fig.write_html(path)
    else:
        fig.show()
    
    


In [62]:
power_plants = df[['name', 'code']].drop_duplicates()

for row in power_plants.itertuples():
    name_plot, code_plot = row.name, row.code
    ds_n_c_plot = Data_selector(Data_selector(df).select_peaks(goodness=1))
    df_n_c_plot = ds_n_c_plot.filter_name_code(name_plot, code_plot)
    try:
        show(df_n_c_plot, save=True, param={"name": name_plot, "code": code_plot})
    except Exception as e:
        print(e)
        print(name_plot,code_plot)